# Notebook 6: Batch Train All Continual SD-LoRA Adapters

Bu notebook, Notebook 2 akisini baz alir ve tanimli adapterleri sirayla egitir.

Akis:
1. Repo bootstrap ve Notebook 2 helper'larini yukle.
2. Adapter matrisi ve sirayi tanimla.
3. Her adapter icin Notebook 2 parametre, dataset validation, training, OOD calibration, export ve final evaluation hucrelerini calistir.
4. Her run icin ozet yazdir; runtime auto-disconnect batch bitene kadar kapali kalir.


In [1]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK6_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)

def _build_repo_access_url(repo_url, token):
    token = str(token or '').strip()
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _build_repo_access_url(REPO_URL, os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN'))
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', clone_url, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK6_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SONRAKI] Parametre ve adapter secimi hucresine gecin; model erisimi ayrica access-check hucresinde dogrulanacak.


In [2]:
NOTEBOOK_NAME = "6_train_all_continual_sd_lora_adapters.ipynb"
NOTEBOOK_FILENAME = "6_train_all_continual_sd_lora_adapters.executed.ipynb"
AUTO_DISCONNECT_RUNTIME = False
AUTO_PUSH_TO_GITHUB = True
NB6_AUTO_DISCONNECT_RUNTIME = True
NB6_AUTO_DISCONNECT_GRACE_SECONDS = 20
ENABLE_BAYESIAN_OPTIMIZATION = True
MANUAL_PARAM_OVERRIDES = {}
DEFAULT_RUNTIME_PARAMS = {
    "AUTO_DISCONNECT_RUNTIME": False,
    "AUTO_PUSH_TO_GITHUB": True,
}
NB6_MANUAL_PARAM_OVERRIDES = {}

from scripts.notebook_helpers.adapter_recommendations import get_adapter_recs
ADAPTER_RECS = get_adapter_recs()

NB6_ADAPTER_SEQUENCE = [
    "apricot__leaf",
    "apricot__fruit",
    "strawberry__leaf",
    "strawberry__fruit",
    "grape__leaf",
    "grape__fruit",
    "tomato__leaf",
    "tomato__fruit",
]


In [ ]:
import json

NB6_RESULTS = {}

for index, adapter_key in enumerate(NB6_ADAPTER_SEQUENCE, start=1):
    print(f"\n[NB6] Starting {index}/{len(NB6_ADAPTER_SEQUENCE)}: {adapter_key}")
    adapter_rec = ADAPTER_RECS[adapter_key]
    CROP_NAME = adapter_rec["crop"]
    PART_NAME = adapter_rec["part"]
    DATASET_NAME = adapter_key
    ADAPTER_KEY = adapter_key
    MANUAL_PARAM_OVERRIDES = dict(NB6_MANUAL_PARAM_OVERRIDES.get(adapter_key, {}))
    try:
        run_cell_script('nb2_cell03_runtime_setup.py', globals())
        run_cell_script('nb2_cell04_parameter_resolution.py', globals())
        run_cell_script('nb2_cell05_access_check.py', globals())
        run_cell_script('nb2_cell06_dataset_validation.py', globals())
        run_cell_script('nb2_cell07_engine_init.py', globals())
        run_cell_script('nb2_cell08_ood_config_verify.py', globals())
        run_cell_script('nb2_cell09_training.py', globals())
        run_cell_script('nb2_cell10_ood_calibration.py', globals())
        run_cell_script('nb2_cell11_adapter_save.py', globals())
        run_cell_script('nb2_cell12_final_evaluation.py', globals())
        NB6_RESULTS[adapter_key] = {
            "status": "ok",
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
        }
    except Exception as exc:
        NB6_RESULTS[adapter_key] = {
            "status": "failed",
            "error": str(exc),
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
        }
        print(f"[NB6] FAILED {adapter_key}: {exc}")
        continue

print('\n[NB6] SUMMARY')
print(json.dumps(NB6_RESULTS, indent=2, ensure_ascii=False))

from scripts.colab_notebook_helpers import maybe_auto_disconnect_colab_runtime

NB6_COMPLETION_REPORT = {
    "ready": True,
    "checks": {
        "batch_loop_completed": True,
        "all_adapters_attempted": len(NB6_RESULTS) == len(NB6_ADAPTER_SEQUENCE),
    },
    "missing": [],
    "soft_missing": [],
    "adapter_results": NB6_RESULTS,
}
maybe_auto_disconnect_colab_runtime(
    enabled=bool(NB6_AUTO_DISCONNECT_RUNTIME),
    grace_period_sec=float(NB6_AUTO_DISCONNECT_GRACE_SECONDS),
    telemetry=globals().get("TELEMETRY"),
    completion_report=NB6_COMPLETION_REPORT,
)



[NB6] Starting 1/8: apricot__leaf
[BOOTSTRAP] run_id=apricot_leaf_2026-05-25_16-51-40 crop=apricot part=leaf
[ADAPTER_SELECTED] key=apricot__leaf crop=apricot part=leaf dataset=apricot__leaf ood=data/prepared_runtime_datasets/apricot__leaf/ood oe_enabled=True oe=data/prepared_runtime_datasets/apricot__leaf/oe run_id=apricot_leaf_2026-05-25_16-51-40
[TRAINING_PLAN] epochs=24 batch=52 lr=0.00012 lora_r=20 allow_under_min=False validation_every=1
[SONRAKI] Erisim kontrolu -> dataset validation -> engine init -> training -> calibration -> save/eval hucresini sirayla calistirin.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=apricot epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=True
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=apricot__leaf
[OOD] ood_root=data/prepared_runtime_datasets/apricot__leaf/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/apricot__leaf/oe ask_for_oe_root=False enabled=True loss_weight=0.3
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1
[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

...

[ENGINE] Hazir. trainable_params=17,313,806  classes=10
[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 4.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 4.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.
[TRAIN] epochs=24 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/58 loss=0.0000 lr=0.000085 speed=261.5s/s elapsed=13s eta=1536s
[TRAIN] epoch=1 batch=24/58 loss=0.0000 lr=0.000085 speed=261.2s/s elapsed=18s eta=998s
[TRAIN] epoch=1 batch=36/58 loss=0.0000 lr=0.000085 speed=261.3s/s elapsed=22s eta=816s
[TRAIN] epoch=1 batch=48/58 loss=0.0000 lr=0.000085 speed=261.3s/s elapsed=26s eta=723s
[CKPT] epoch

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[BOOTSTRAP] run_id=tomato_fruit_2026-05-25_17-31-02 crop=tomato part=fruit
[ADAPTER_SELECTED] key=tomato__fruit crop=tomato part=fruit dataset=tomato__fruit ood=data/prepared_runtime_datasets/tomato__fruit/ood oe_enabled=True oe=data/prepared_runtime_datasets/tomato__fruit/oe run_id=tomato_fruit_2026-05-25_17-31-02
[TRAINING_PLAN] epochs=30 batch=64 lr=6e-05 lora_r=24 allow_under_min=False validation_every=1
[SONRAKI] Erisim kontrolu -> dataset validation -> engine init -> training -> calibration -> save/eval hucresini sirayla calistirin.
[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=tomato epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=True
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=tomato__fruit
[OOD] ood_root=data/prepared_runtime_datasets/tomato__fruit/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runt

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=13,772,555  classes=7
[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.
[TRAIN] epochs=30 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/35 loss=0.0000 lr=0.000060 speed=243.3s/s elapsed=11s eta=921s
[TRAIN] epoch=1 batch=24/35 loss=0.0000 lr=0.000060 speed=253.5s/s elapsed=14s eta=591s
[CKPT] epoch_end epoch=1 step=35
[EPOCH] 1/30: train_loss=1.9632 val_loss=1.9133 val_acc=0.2773 macro_f1=0.0720 * BEST
[TRAIN] epoch=2 batch=12/35 loss=0.0000 lr=0.000060 speed=197.8s/s elap